# Travel Efficiency Explorer

Single-run analysis of ambulance transport efficiency:

1. **Unserved requests** — how many times a T1/T2/T3 requested transport but no ambulance was available
2. **Waiting time boxplots** — distribution of transport wait times by triage and phase

In [ ]:
import sys
from pathlib import Path

_root = Path(".").resolve()
while not (_root / "paths.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))
import paths

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
FOLDER_NAME = "FOLDER_NAME"
run_folder = paths.SINGLE_RUN_OUTPUT / FOLDER_NAME

wt = pd.read_excel(run_folder / "wait_times.xlsx")
c2_ccp = pd.read_excel(run_folder / "c2_ccp_r1.xlsx")
c2_r1  = pd.read_excel(run_folder / "c2_r1_r2.xlsx")

# Short triage label: T1_BCAS -> T1
wt["triage_short"] = wt["triage"].str.split("_").str[0]

print(f"Loaded {len(wt)} wait-time records")
print(f"C2 CCP->R1: {len(c2_ccp)} dispatch events")
print(f"C2 R1->R2:  {len(c2_r1)} dispatch events")

## 1 — Unserved transport requests

At each C2 dispatch event we know how many requests came in and how many
were assigned. The difference is the number of patients left waiting
because no ambulance was idle. We count these per triage level.

In [ ]:
def count_unserved(c2_df, label):
    """Parse C2 logs and count unserved requests per triage."""
    records = []
    for _, row in c2_df.iterrows():
        reqs = json.loads(row["transport_requests"])
        assignments = json.loads(row["assignments"])
        assigned_ids = {pid for pids in assignments.values() for pid in pids}

        for req in reqs:
            pid = req["casualty_id"]
            triage = req["triage"].split("_")[0] if "_" in req["triage"] else req["triage"]
            records.append({
                "time": row["time"],
                "triage": triage,
                "served": pid in assigned_ids,
                "c2": label,
            })
    return pd.DataFrame(records)


req_ccp = count_unserved(c2_ccp, "CCP->R1")
req_r1  = count_unserved(c2_r1,  "R1->R2")
reqs = pd.concat([req_ccp, req_r1], ignore_index=True)

for c2_label in ["CCP->R1", "R1->R2"]:
    sub = reqs[reqs["c2"] == c2_label]
    if sub.empty:
        print(f"\n{c2_label}: no dispatch events")
        continue
    total    = sub.groupby("triage")["served"].count()
    unserved = sub.groupby("triage")["served"].apply(lambda s: (~s).sum())
    summary  = pd.DataFrame({"total_requests": total, "unserved": unserved})
    summary["unserved_pct"] = (summary["unserved"] / summary["total_requests"] * 100).round(1)
    print(f"\n{c2_label} dispatch requests:")
    print(summary.to_string())

In [ ]:
# ── Bar chart: unserved requests per triage and C2 ────────────────────────
TRIAGE_ORDER  = ["T1", "T2", "T3"]
TRIAGE_COLORS = {"T1": "#e74c3c", "T2": "#f39c12", "T3": "#3498db"}

for c2_label in ["CCP->R1", "R1->R2"]:
    sub = reqs[(reqs["c2"] == c2_label) & (~reqs["served"])]
    if sub.empty:
        print(f"{c2_label}: no unserved requests")
        continue
    counts = sub.groupby("triage").size().reindex(TRIAGE_ORDER).fillna(0)

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(TRIAGE_ORDER, counts,
                  color=[TRIAGE_COLORS[t] for t in TRIAGE_ORDER],
                  edgecolor="black", width=0.5)
    for bar, val in zip(bars, counts):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                    str(int(val)), ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_ylabel("Unserved requests")
    ax.set_xlabel("Triage")
    ax.set_title(f"{c2_label} — requests with no ambulance available")
    ax.set_ylim(bottom=0)
    plt.tight_layout()
    plt.show()

## 2 — Transport waiting time boxplots

Distribution of how long patients waited for an ambulance to arrive,
split by triage level and phase (CCP transport vs R1 transport).

In [ ]:
transport_waits = wt[wt["wait_type"] == "transport"].copy()

PHASE_LABELS = {"CCP": "CCP transport", "Role1": "R1 transport"}

for fac_type, phase_label in PHASE_LABELS.items():
    sub = transport_waits[transport_waits["facility_type"] == fac_type]
    if sub.empty:
        print(f"{phase_label}: no transport wait records")
        continue

    # Group data by triage for boxplot
    data = [sub.loc[sub["triage_short"] == t, "duration"].dropna().values
            for t in TRIAGE_ORDER]
    # Only include triages that have data
    labels = [t for t, d in zip(TRIAGE_ORDER, data) if len(d) > 0]
    data   = [d for d in data if len(d) > 0]

    if not data:
        print(f"{phase_label}: no data after filtering")
        continue

    fig, ax = plt.subplots(figsize=(6, 5))
    bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.5)
    for patch, label in zip(bp["boxes"], labels):
        patch.set_facecolor(TRIAGE_COLORS[label])
        patch.set_alpha(0.7)

    # Annotate median values
    for i, (d, label) in enumerate(zip(data, labels)):
        med = np.median(d)
        ax.text(i + 1, med, f"  {med:.1f}", va="center", fontsize=9, fontweight="bold")

    ax.set_ylabel("Wait time (minutes)")
    ax.set_xlabel("Triage")
    ax.set_title(f"{phase_label} — waiting time for ambulance")
    ax.set_ylim(bottom=0)
    plt.tight_layout()
    plt.show()

    # Print summary stats
    for label, d in zip(labels, data):
        print(f"  {phase_label} {label}: n={len(d):4d}  "
              f"median={np.median(d):.1f}  mean={np.mean(d):.1f}  "
              f"p95={np.percentile(d, 95):.1f}  max={np.max(d):.1f}")